In [57]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.regression.mixed_linear_model import MixedLM
from memoria_icm.datos import *
from memoria_icm.funciones import *
from memoria_icm.graficos import *

In [58]:
df, df_resumen = cargar_datos()
df

Datos cargados. Trackeo: (44324, 28), Resumen: (192, 42)


,index,id,t,x,y,id_ubicacion,especie,tratamiento,salinidad,nombre_archivo,...,a,r,dr,theta_r,v_angular,a_angular,theta_v,curvatura,duracion_video_recortada,nivel_salinidad
0,0,0,0.000000,1.976667,-0.624416,1,CC,E,35-N,Video_001,...,2.047467,0.000000,0.098320,0.000000,-12.040794,-51.274027,-0.881077,0.000000,11.515,normal
1,1,0,0.073174,1.981245,-0.629966,1,CC,E,35-N,Video_001,...,0.763748,0.007194,0.138270,-0.881077,-15.792735,60.562228,-2.033736,126.445970,11.515,normal
2,2,0,0.123811,1.972938,-0.639559,1,CC,E,35-N,Video_001,...,1.150466,0.015596,0.088876,-1.812237,-8.807270,142.715592,-2.177735,147.635591,11.515,normal
3,3,0,0.181472,1.974790,-0.639965,1,CC,E,35-N,Video_001,...,1.312186,0.015662,0.027576,-1.690928,-0.265367,107.160874,-1.927719,764.513321,11.515,normal
4,4,0,0.224938,1.972984,-0.641756,1,CC,E,35-N,Video_001,...,1.431348,0.017727,0.071411,-1.780087,3.049804,56.515627,-0.930846,305.291105,11.515,normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44319,44319,320,11.291740,2.965504,-3.120556,3,LA,R,35,Video_006,...,0.245230,1.719952,0.126393,-2.648078,0.006991,0.096506,-2.552953,16.924950,11.515,normal
44320,44320,320,11.348250,2.959012,-3.125446,3,LA,R,35,Video_006,...,0.681606,1.727987,0.101667,-2.647365,0.021723,0.140003,-2.293583,78.380508,11.515,normal
44321,44321,320,11.393450,2.957334,-3.128929,3,LA,R,35,Video_006,...,1.167128,1.731117,0.169209,-2.646054,0.023688,-0.067894,-2.407994,75.171379,11.515,normal
44322,44322,320,11.429330,2.950067,-3.134238,3,LA,R,35,Video_006,...,0.186546,1.740035,0.201462,-2.645356,0.018080,-0.110120,-2.490801,11.765173,11.515,normal


Este cuadernillo analiza la existencia de diferencias significativas entre los grupos del mismo nivel de salinidad, agrupados por especie. EXPOSICIÓN.

In [59]:
df, df_resumen = cargar_datos() #estos son los definitivos

columnas_cce = [
    'r_centrado_promedio',
    'r_centrado_mediana',
    'dr_centrado_promedio',
    'dr_centrado_mediana'
]

columnas_lae = [
    'a_promedio',
    'a_mediana',
    'dr_centrado_promedio',
    'dr_centrado_mediana',
    'a_angular_centrado_mediana',
    'theta_v_promedio'
]

Datos cargados. Trackeo: (44324, 28), Resumen: (192, 42)


In [60]:
#grupos
cc_baja = df_resumen[
    (df_resumen['nivel_salinidad'] == 'baja') &
    (df_resumen['especie'] == 'CC') &
    (df_resumen['tratamiento'] == 'E')]

cc_normal = df_resumen[
    (df_resumen['nivel_salinidad'] == 'normal') &
    (df_resumen['especie'] == 'CC') &
    (df_resumen['tratamiento'] == 'E')]

cc_alta = df_resumen[
    (df_resumen['nivel_salinidad'] == 'alta') &
    (df_resumen['especie'] == 'CC') &
    (df_resumen['tratamiento'] == 'E')]

la_baja = df_resumen[
    (df_resumen['nivel_salinidad'] == 'baja') &
    (df_resumen['especie'] == 'LA') &
    (df_resumen['tratamiento'] == 'E')]

la_normal = df_resumen[
    (df_resumen['nivel_salinidad'] == 'normal') &
    (df_resumen['especie'] == 'LA') &
    (df_resumen['tratamiento'] == 'E')]

la_alta = df_resumen[
    (df_resumen['nivel_salinidad'] == 'alta') &
    (df_resumen['especie'] == 'LA') &
    (df_resumen['tratamiento'] == 'E')]

In [61]:
locos = {'cc_baja': cc_baja, 'cc_normal': cc_normal, 'cc_alta': cc_alta}
erizos = {'la_baja': la_baja, 'la_normal': la_normal, 'la_alta': la_alta}
resultados_cc = {}
resultados_la = {}
factor = 'salinidad'

#cc
for nombre_grupo, df in locos.items():
    clave = f"{nombre_grupo}"
    resultados_cc[clave] = pd.concat([testear_supuestos(df, var, factor) for var in columnas_cce],
                                  ignore_index=True)
    
df_resultados_cc = pd.concat([pd.DataFrame(lista).assign(grupo=clave) for clave, lista in resultados_cc.items()],
                          ignore_index=True)

#la
for nombre_grupo, df in erizos.items():
    clave = f"{nombre_grupo}"
    resultados_la[clave] = pd.concat([testear_supuestos(df, var, factor) for var in columnas_lae],
                                  ignore_index=True)
    
df_resultados_la = pd.concat([pd.DataFrame(lista).assign(grupo=clave) for clave, lista in resultados_la.items()],
                          ignore_index=True)

#este dataframe entrega resultados de TODOS LOS GRUPOS y TODAS LAS variables_resumen_resumen de una sola vez,
#luego filtro

# LOCOS

## baja

### Análisis de la varianza

In [62]:
resultados_cc_baja = df_resultados_cc[(df_resultados_cc['grupo'] == 'cc_baja')]

In [63]:
variables_resumen_por_test = (resultados_cc_baja[['variable', 'test_sugerido']].drop_duplicates()
    .groupby('test_sugerido')['variable']
    .unique()
    .to_dict())

print("LOCOS BAJA - QUÉ APLICAR SEGÚN SUPUESTOS")
for tipo, lista in variables_resumen_por_test.items():
    print(f"Puedo aplicar {tipo} en: {list(lista)}")
    
listas_por_test = {
    tipo: grupo['variable'].drop_duplicates().tolist()
    for tipo, grupo in resultados_cc_baja.dropna(subset=['test_sugerido']).groupby('test_sugerido')}

LOCOS BAJA - QUÉ APLICAR SEGÚN SUPUESTOS
Puedo aplicar ANOVA en: ['r_centrado_promedio', 'r_centrado_mediana']
Puedo aplicar Kruskal-Wallis en: ['dr_centrado_promedio', 'dr_centrado_mediana']


In [64]:
df_efectos_cc_baja = analizar_grupo(cc_baja, listas_por_test, factor)

pvalues_significativos_cc_baja = df_efectos_cc_baja.loc[(df_efectos_cc_baja['p_value'] < 0.05)]
variables_candidatas_cc_baja = pvalues_significativos_cc_baja['variable'].tolist()
pvalues_significativos_cc_baja

,variable,test,p_value,η²,ω²/ε²


### Pruebas post hoc

In [65]:
variables_tukey = list(df_efectos_cc_baja.loc[
    (df_efectos_cc_baja['test'] == 'ANOVA') &
    (df_efectos_cc_baja['variable'].isin(variables_candidatas_cc_baja)),
    'variable'])

variables_gh = list(df_efectos_cc_baja.loc[
    (df_efectos_cc_baja['test'] == "Welch's ANOVA") &
    (df_efectos_cc_baja['variable'].isin(variables_candidatas_cc_baja)),
    'variable'])

variables_dunn = list(df_efectos_cc_baja.loc[
    (df_efectos_cc_baja['test'] == 'Kruskal-Wallis') &
    (df_efectos_cc_baja['variable'].isin(variables_candidatas_cc_baja)),
    'variable'])

In [66]:
if variables_tukey:
    tukey_df = posthoc_anova_tukey(cc_baja, variables_tukey, factor)
    
if variables_gh:
    gh_df = posthoc_welch_gameshowell(cc_baja, variables_gh, factor)
    
if variables_dunn:
    dunn_df = posthoc_kruskal_dunn(cc_baja, variables_dunn, factor)

In [67]:
if 'tukey_df' in locals():
    posthocs_tukey = {variable: tukey_df.loc[tukey_df['variable'] == variable] for variable in variables_tukey}
    print("Hay df con post hoc para Tukey. Revisar")
    
if 'gh_df' in locals():
    posthocs_gh = {variable: gh_df.loc[gh_df['variable'] == variable] for variable in variables_gh}
    print("Hay df con post hoc para GH. Revisar")
    
if 'dunn_df' in locals():
    posthocs_dunn = {variable: dunn_df.loc[dunn_df['variable'] == variable] for variable in variables_dunn}
    print("Hay df con post hoc para Dunn. Revisar")

Hay df con post hoc para Tukey. Revisar
Hay df con post hoc para GH. Revisar
Hay df con post hoc para Dunn. Revisar


In [68]:
# print(posthocs_tukey['a_angular_promedio'])

## normal

In [69]:
resultados_cc_normal = df_resultados_cc[(df_resultados_cc['grupo'] == 'cc_normal')]
variables_resumen_por_test = (resultados_cc_normal[['variable', 'test_sugerido']].drop_duplicates()
    .groupby('test_sugerido')['variable']
    .unique()
    .to_dict())

print("LOCOS NORMAL - QUÉ APLICAR SEGÚN SUPUESTOS")
for tipo, lista in variables_resumen_por_test.items():
    print(f"Puedo aplicar {tipo} en: {list(lista)}")
    
listas_por_test = {
    tipo: grupo['variable'].drop_duplicates().tolist()
    for tipo, grupo in resultados_cc_normal.dropna(subset=['test_sugerido']).groupby('test_sugerido')}
df_efectos_cc_normal = analizar_grupo(cc_normal, listas_por_test, factor)

pvalues_significativos_cc_normal = df_efectos_cc_normal.loc[(df_efectos_cc_normal['p_value'] < 0.05)]
variables_candidatas_cc_normal = pvalues_significativos_cc_normal['variable'].tolist()
#variables_candidatas_cc_normal #podemos mostrar el dataframe completo también
print(pvalues_significativos_cc_normal)


LOCOS NORMAL - QUÉ APLICAR SEGÚN SUPUESTOS
Puedo aplicar Kruskal-Wallis en: ['r_centrado_promedio', 'r_centrado_mediana', 'dr_centrado_promedio', 'dr_centrado_mediana']
Empty DataFrame
Columns: [variable, test, p_value, η², ω²/ε²]
Index: []


In [70]:
### Pruebas post hoc
variables_tukey = list(df_efectos_cc_normal.loc[
    (df_efectos_cc_normal['test'] == 'ANOVA') &
    (df_efectos_cc_normal['variable'].isin(variables_candidatas_cc_normal)),
    'variable'])

variables_gh = list(df_efectos_cc_normal.loc[
    (df_efectos_cc_normal['test'] == "Welch's ANOVA") &
    (df_efectos_cc_normal['variable'].isin(variables_candidatas_cc_normal)),
    'variable'])

variables_dunn = list(df_efectos_cc_normal.loc[
    (df_efectos_cc_normal['test'] == 'Kruskal-Wallis') &
    (df_efectos_cc_normal['variable'].isin(variables_candidatas_cc_normal)),
    'variable'])
if variables_tukey:
    tukey_df = posthoc_anova_tukey(cc_normal, variables_tukey, factor)
    
if variables_gh:
    gh_df = posthoc_welch_gameshowell(cc_normal, variables_gh, factor)
    
if variables_dunn:
    dunn_df = posthoc_kruskal_dunn(cc_normal, variables_dunn, factor)
if 'tukey_df' in locals():
    posthocs_tukey = {variable: tukey_df.loc[tukey_df['variable'] == variable] for variable in variables_tukey}
    print("Hay df con post hoc para Tukey. Revisar")
    
if 'gh_df' in locals():
    posthocs_gh = {variable: gh_df.loc[gh_df['variable'] == variable] for variable in variables_gh}
    print("Hay df con post hoc para GH. Revisar")
    
if 'dunn_df' in locals():
    posthocs_dunn = {variable: dunn_df.loc[dunn_df['variable'] == variable] for variable in variables_dunn}
    print("Hay df con post hoc para Dunn. Revisar")

Hay df con post hoc para Tukey. Revisar
Hay df con post hoc para GH. Revisar
Hay df con post hoc para Dunn. Revisar


In [71]:
pvalues_significativos_cc_normal

,variable,test,p_value,η²,ω²/ε²


## alta

In [72]:
resultados_cc_alta = df_resultados_cc[(df_resultados_cc['grupo'] == 'cc_alta')]
variables_resumen_por_test = (resultados_cc_alta[['variable', 'test_sugerido']].drop_duplicates()
    .groupby('test_sugerido')['variable']
    .unique()
    .to_dict())

print("LOCOS ALTA - QUÉ APLICAR SEGÚN SUPUESTOS")
for tipo, lista in variables_resumen_por_test.items():
    print(f"Puedo aplicar {tipo} en: {list(lista)}")
    
listas_por_test = {
    tipo: grupo['variable'].drop_duplicates().tolist()
    for tipo, grupo in resultados_cc_alta.dropna(subset=['test_sugerido']).groupby('test_sugerido')}
df_efectos_cc_alta = analizar_grupo(cc_alta, listas_por_test, factor)

pvalues_significativos_cc_alta = df_efectos_cc_alta.loc[(df_efectos_cc_alta['p_value'] < 0.05)]
variables_candidatas_cc_alta = pvalues_significativos_cc_alta['variable'].tolist()
print(pvalues_significativos_cc_alta)

LOCOS ALTA - QUÉ APLICAR SEGÚN SUPUESTOS
Puedo aplicar ANOVA en: ['r_centrado_promedio', 'r_centrado_mediana', 'dr_centrado_mediana']
Puedo aplicar Kruskal-Wallis en: ['dr_centrado_promedio']
Empty DataFrame
Columns: [variable, test, p_value, η², ω²/ε²]
Index: []


In [73]:

variables_tukey = list(df_efectos_cc_alta.loc[
    (df_efectos_cc_alta['test'] == 'ANOVA') &
    (df_efectos_cc_alta['variable'].isin(variables_candidatas_cc_alta)),
    'variable'])

variables_gh = list(df_efectos_cc_alta.loc[
    (df_efectos_cc_alta['test'] == "Welch's ANOVA") &
    (df_efectos_cc_alta['variable'].isin(variables_candidatas_cc_alta)),
    'variable'])

variables_dunn = list(df_efectos_cc_alta.loc[
    (df_efectos_cc_alta['test'] == 'Kruskal-Wallis') &
    (df_efectos_cc_alta['variable'].isin(variables_candidatas_cc_alta)),
    'variable'])

if variables_tukey:
    tukey_df = posthoc_anova_tukey(cc_alta, variables_tukey, factor)
if variables_gh:
    gh_df = posthoc_welch_gameshowell(cc_alta, variables_gh, factor)
if variables_dunn:
    dunn_df = posthoc_kruskal_dunn(cc_alta, variables_dunn, factor)

if 'tukey_df' in locals():
    posthocs_tukey = {variable: tukey_df.loc[tukey_df['variable'] == variable] for variable in variables_tukey}
    print("Hay df con post hoc para Tukey. Revisar")
if 'gh_df' in locals():
    posthocs_gh = {variable: gh_df.loc[gh_df['variable'] == variable] for variable in variables_gh}
    print("Hay df con post hoc para GH. Revisar")
if 'dunn_df' in locals():
    posthocs_dunn = {variable: dunn_df.loc[dunn_df['variable'] == variable] for variable in variables_dunn}
    print("Hay df con post hoc para Dunn. Revisar")


Hay df con post hoc para Tukey. Revisar
Hay df con post hoc para GH. Revisar
Hay df con post hoc para Dunn. Revisar


In [74]:
print(posthocs_tukey)

{}


# ERIZOS

## baja

In [75]:
resultados_la_baja = df_resultados_la[(df_resultados_la['grupo'] == 'la_baja')]
variables_resumen_por_test = (resultados_la_baja[['variable', 'test_sugerido']].drop_duplicates()
    .groupby('test_sugerido')['variable']
    .unique()
    .to_dict())

print("ERIZOS BAJA - QUÉ APLICAR SEGÚN SUPUESTOS")
for tipo, lista in variables_resumen_por_test.items():
    print(f"Puedo aplicar {tipo} en: {list(lista)}")
    
listas_por_test = {
    tipo: grupo['variable'].drop_duplicates().tolist()
    for tipo, grupo in resultados_la_baja.dropna(subset=['test_sugerido']).groupby('test_sugerido')}
df_efectos_la_baja = analizar_grupo(la_baja, listas_por_test, factor)

pvalues_significativos_la_baja = df_efectos_la_baja.loc[(df_efectos_la_baja['p_value'] < 0.05)]
variables_candidatas_la_baja = pvalues_significativos_la_baja['variable'].tolist()
print(pvalues_significativos_la_baja)


ERIZOS BAJA - QUÉ APLICAR SEGÚN SUPUESTOS
Puedo aplicar ANOVA en: ['a_angular_centrado_mediana', 'theta_v_promedio']
Puedo aplicar Kruskal-Wallis en: ['a_mediana', 'dr_centrado_promedio']
Puedo aplicar Welch's ANOVA en: ['a_promedio', 'dr_centrado_mediana']
               variable            test   p_value        η²     ω²/ε²
2            a_promedio   Welch's ANOVA  0.000023  0.730404  0.675356
0             a_mediana  Kruskal-Wallis  0.003057  0.661472  0.605051
1  dr_centrado_promedio  Kruskal-Wallis  0.005436  0.602786  0.536583


In [76]:

variables_tukey = list(df_efectos_la_baja.loc[
    (df_efectos_la_baja['test'] == 'ANOVA') &
    (df_efectos_la_baja['variable'].isin(variables_candidatas_la_baja)),
    'variable'])

variables_gh = list(df_efectos_la_baja.loc[
    (df_efectos_la_baja['test'] == "Welch's ANOVA") &
    (df_efectos_la_baja['variable'].isin(variables_candidatas_la_baja)),
    'variable'])

variables_dunn = list(df_efectos_la_baja.loc[
    (df_efectos_la_baja['test'] == 'Kruskal-Wallis') &
    (df_efectos_la_baja['variable'].isin(variables_candidatas_la_baja)),
    'variable'])

if variables_tukey:
    tukey_df = posthoc_anova_tukey(la_baja, variables_tukey, factor)
if variables_gh:
    gh_df = posthoc_welch_gameshowell(la_baja, variables_gh, factor)
if variables_dunn:
    dunn_df = posthoc_kruskal_dunn(la_baja, variables_dunn, factor)

if 'tukey_df' in locals():
    posthocs_tukey = {variable: tukey_df.loc[tukey_df['variable'] == variable] for variable in variables_tukey}
    print("Hay df con post hoc para Tukey. Revisar")
if 'gh_df' in locals():
    posthocs_gh = {variable: gh_df.loc[gh_df['variable'] == variable] for variable in variables_gh}
    print("Hay df con post hoc para GH. Revisar")
if 'dunn_df' in locals():
    posthocs_dunn = {variable: dunn_df.loc[dunn_df['variable'] == variable] for variable in variables_dunn}
    print("Hay df con post hoc para Dunn. Revisar")


Hay df con post hoc para Tukey. Revisar
Hay df con post hoc para GH. Revisar
Hay df con post hoc para Dunn. Revisar


In [77]:
print(posthocs_dunn)

{'a_mediana':    grupo1 grupo2   p-valor   variable
0      15     15  1.000000  a_mediana
1      20     15  0.460048  a_mediana
2      25     15  0.026433  a_mediana
3      30     15  0.005849  a_mediana
4      15     20  0.460048  a_mediana
5      20     20  1.000000  a_mediana
6      25     20  0.300157  a_mediana
7      30     20  0.125200  a_mediana
8      15     25  0.026433  a_mediana
9      20     25  0.300157  a_mediana
10     25     25  1.000000  a_mediana
11     30     25  0.626269  a_mediana
12     15     30  0.005849  a_mediana
13     20     30  0.125200  a_mediana
14     25     30  0.626269  a_mediana
15     30     30  1.000000  a_mediana, 'dr_centrado_promedio':    grupo1 grupo2   p-valor              variable
16     15     15  1.000000  dr_centrado_promedio
17     20     15  0.495492  dr_centrado_promedio
18     25     15  0.347279  dr_centrado_promedio
19     30     15  0.002869  dr_centrado_promedio
20     15     20  0.495492  dr_centrado_promedio
21     20     20  1.0

## normal

In [78]:
resultados_la_normal = df_resultados_la[(df_resultados_la['grupo'] == 'la_normal')]
variables_resumen_por_test = (resultados_la_normal[['variable', 'test_sugerido']].drop_duplicates()
    .groupby('test_sugerido')['variable']
    .unique()
    .to_dict())

print("ERIZOS NORMAL - QUÉ APLICAR SEGÚN SUPUESTOS")
for tipo, lista in variables_resumen_por_test.items():
    print(f"Puedo aplicar {tipo} en: {list(lista)}")
    
listas_por_test = {
    tipo: grupo['variable'].drop_duplicates().tolist()
    for tipo, grupo in resultados_la_normal.dropna(subset=['test_sugerido']).groupby('test_sugerido')}
df_efectos_la_normal = analizar_grupo(la_normal, listas_por_test, factor)

pvalues_significativos_la_normal = df_efectos_la_normal.loc[(df_efectos_la_normal['p_value'] < 0.05)]
variables_candidatas_la_normal = pvalues_significativos_la_normal['variable'].tolist()
pvalues_significativos_la_normal

ERIZOS NORMAL - QUÉ APLICAR SEGÚN SUPUESTOS
Puedo aplicar ANOVA en: ['theta_v_promedio']
Puedo aplicar Kruskal-Wallis en: ['a_promedio', 'a_mediana']
Puedo aplicar Welch's ANOVA en: ['dr_centrado_promedio', 'dr_centrado_mediana', 'a_angular_centrado_mediana']


,variable,test,p_value,η²,ω²/ε²


In [79]:

variables_tukey = list(df_efectos_la_normal.loc[
    (df_efectos_la_normal['test'] == 'ANOVA') &
    (df_efectos_la_normal['variable'].isin(variables_candidatas_la_normal)),
    'variable'])

variables_gh = list(df_efectos_la_normal.loc[
    (df_efectos_la_normal['test'] == "Welch's ANOVA") &
    (df_efectos_la_normal['variable'].isin(variables_candidatas_la_normal)),
    'variable'])

variables_dunn = list(df_efectos_la_normal.loc[
    (df_efectos_la_normal['test'] == 'Kruskal-Wallis') &
    (df_efectos_la_normal['variable'].isin(variables_candidatas_la_normal)),
    'variable'])

if variables_tukey:
    tukey_df = posthoc_anova_tukey(la_normal, variables_tukey, factor)
if variables_gh:
    gh_df = posthoc_welch_gameshowell(la_normal, variables_gh, factor)
if variables_dunn:
    dunn_df = posthoc_kruskal_dunn(la_normal, variables_dunn, factor)

if 'tukey_df' in locals():
    posthocs_tukey = {variable: tukey_df.loc[tukey_df['variable'] == variable] for variable in variables_tukey}
    print("Hay df con post hoc para Tukey. Revisar")
if 'gh_df' in locals():
    posthocs_gh = {variable: gh_df.loc[gh_df['variable'] == variable] for variable in variables_gh}
    print("Hay df con post hoc para GH. Revisar")
if 'dunn_df' in locals():
    posthocs_dunn = {variable: dunn_df.loc[dunn_df['variable'] == variable] for variable in variables_dunn}
    print("Hay df con post hoc para Dunn. Revisar")


Hay df con post hoc para Tukey. Revisar
Hay df con post hoc para GH. Revisar
Hay df con post hoc para Dunn. Revisar


## alta

In [80]:
resultados_la_alta = df_resultados_la[(df_resultados_la['grupo'] == 'la_alta')]
variables_resumen_por_test = (resultados_la_alta[['variable', 'test_sugerido']].drop_duplicates()
    .groupby('test_sugerido')['variable']
    .unique()
    .to_dict())

print("ERIZOS ALTA - QUÉ APLICAR SEGÚN SUPUESTOS")
for tipo, lista in variables_resumen_por_test.items():
    print(f"Puedo aplicar {tipo} en: {list(lista)}")
    
listas_por_test = {
    tipo: grupo['variable'].drop_duplicates().tolist()
    for tipo, grupo in resultados_la_alta.dropna(subset=['test_sugerido']).groupby('test_sugerido')}
df_efectos_la_alta = analizar_grupo(la_alta, listas_por_test, factor)

pvalues_significativos_la_alta = df_efectos_la_alta.loc[(df_efectos_la_alta['p_value'] < 0.05)]
variables_candidatas_la_alta = pvalues_significativos_la_alta['variable'].tolist()
print(pvalues_significativos_la_alta)


ERIZOS ALTA - QUÉ APLICAR SEGÚN SUPUESTOS
Puedo aplicar ANOVA en: ['a_promedio', 'a_mediana', 'dr_centrado_promedio', 'a_angular_centrado_mediana', 'theta_v_promedio']
Puedo aplicar Welch's ANOVA en: ['dr_centrado_mediana']


     variable   test   p_value        η²     ω²/ε²
2   a_mediana  ANOVA  0.009909  0.602616  0.503145
1  a_promedio  ANOVA  0.019643  0.544338  0.433454


In [81]:

variables_tukey = list(df_efectos_la_alta.loc[
    (df_efectos_la_alta['test'] == 'ANOVA') &
    (df_efectos_la_alta['variable'].isin(variables_candidatas_la_alta)),
    'variable'])

variables_gh = list(df_efectos_la_alta.loc[
    (df_efectos_la_alta['test'] == "Welch's ANOVA") &
    (df_efectos_la_alta['variable'].isin(variables_candidatas_la_alta)),
    'variable'])

variables_dunn = list(df_efectos_la_alta.loc[
    (df_efectos_la_alta['test'] == 'Kruskal-Wallis') &
    (df_efectos_la_alta['variable'].isin(variables_candidatas_la_alta)),
    'variable'])

if variables_tukey:
    tukey_df = posthoc_anova_tukey(la_alta, variables_tukey, factor)
if variables_gh:
    gh_df = posthoc_welch_gameshowell(la_alta, variables_gh, factor)
if variables_dunn:
    dunn_df = posthoc_kruskal_dunn(la_alta, variables_dunn, factor)

if 'tukey_df' in locals():
    posthocs_tukey = {variable: tukey_df.loc[tukey_df['variable'] == variable] for variable in variables_tukey}
    print("Hay df con post hoc para Tukey. Revisar")
if 'gh_df' in locals():
    posthocs_gh = {variable: gh_df.loc[gh_df['variable'] == variable] for variable in variables_gh}
    print("Hay df con post hoc para GH. Revisar")
if 'dunn_df' in locals():
    posthocs_dunn = {variable: dunn_df.loc[dunn_df['variable'] == variable] for variable in variables_dunn}
    print("Hay df con post hoc para Dunn. Revisar")

Hay df con post hoc para Tukey. Revisar
Hay df con post hoc para GH. Revisar
Hay df con post hoc para Dunn. Revisar


In [82]:
print(posthocs_tukey)

{'a_mediana':   group1 group2  meandiff   p-adj   lower   upper  reject   variable
0     40     45   -0.0066  0.9991 -0.4659  0.4527   False  a_mediana
1     40     50   -0.4791  0.0283 -0.9043 -0.0538    True  a_mediana
2     45     50   -0.4725  0.0188 -0.8607 -0.0842    True  a_mediana, 'a_promedio':   group1 group2  meandiff   p-adj   lower   upper  reject    variable
3     40     45    0.0015  1.0000 -0.6568  0.6598   False  a_promedio
4     40     50   -0.6037  0.0522 -1.2132  0.0057   False  a_promedio
5     45     50   -0.6053  0.0337 -1.1617 -0.0489    True  a_promedio}
